In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
url = "https://raw.githubusercontent.com/premlatamd/week03/refs/heads/main/hotel_bookings.csv"
df = pd.read_csv(url)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


#R1

In [3]:
df.duplicated().sum()

np.int64(31994)

In [4]:
df.drop_duplicates(inplace=True)
df.duplicated().sum(),df.shape

(np.int64(0), (87396, 32))

In [5]:
pd.crosstab(
    df['is_repeated_guest'],
    df['is_canceled'],
    normalize='index'
)*100

is_canceled,0,1
is_repeated_guest,,
0,71.703123,28.296877
1,92.357247,7.642753


#R2

##*1. The data*

In [6]:
#grouping variables and one numeric variable
for col in df.columns:
  print(col,df[col].unique())

hotel ['Resort Hotel' 'City Hotel']
is_canceled [0 1]
lead_time [342 737   7  13  14   0   9  85  75  23  35  68  18  37  12  72 127  78
  48  60  77  99 118  95  96  69  45  40  15  36  43  70  16 107  47 113
  90  50  93  76   3   1  10   5  17  51  71  63  62 101   2  81 368 364
 324  79  21 109 102   4  98  92  26  73 115  86  52  29  30  33  32   8
 100  44  80  97  64  39  34  27  82  94 110 111  84  66 104  28 258 112
  65  67  55  88  54 292  83 105 280 394  24 103 366 249  22  91  11 108
 106  31  87  41 304 117  59  53  58 116  42 321  38  56  49 317   6  57
  19  25 315 123  46  89  61 312 299 130  74 298 119  20 286 136 129 124
 327 131 460 140 114 139 122 137 126 120 128 135 150 143 151 132 125 157
 147 138 156 164 346 159 160 161 333 381 149 154 297 163 314 155 323 340
 356 142 328 144 336 248 302 175 344 382 146 170 166 338 167 310 148 165
 172 171 145 121 178 305 173 152 354 347 158 185 349 183 352 177 200 192
 361 207 174 330 134 350 334 283 153 197 133 241 193 235 194

##*2. Quality Audit*

In [7]:
#after Duplicate data remove
df.shape

(87396, 32)

In [8]:
#missing values
df.isna().sum().sort_values(ascending=False)

,0
company,82137
agent,12193
country,452
children,4
arrival_date_month,0
arrival_date_week_number,0
hotel,0
is_canceled,0
stays_in_weekend_nights,0
arrival_date_day_of_month,0


In [9]:
#containng abnorma values
df["meal"].unique()

array(['BB', 'FB', 'HB', 'SC', 'Undefined'], dtype=object)

In [10]:
df["market_segment"].unique()

array(['Direct', 'Corporate', 'Online TA', 'Offline TA/TO',
       'Complementary', 'Groups', 'Undefined', 'Aviation'], dtype=object)

In [11]:
df["distribution_channel"].unique()

array(['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS'], dtype=object)

##*3. Suspicious Values*

In [12]:
#1. negetive adr
(df['adr'] < 0).sum()

np.int64(1)

In [13]:
#2. company column has huge missing value which is not good for model
df["company"].isna().sum()

np.int64(82137)

In [14]:
#3. presence of outliers ,as mean =106.337246 and max	= 5400.000000
df["adr"].describe()

,adr
count,87396.000000
mean,106.337246
std,55.013953
min,-6.380000
25%,72.000000
50%,98.100000
75%,134.000000
max,5400.000000


#R3

In [15]:
group_stats = df.groupby('is_repeated_guest')['is_canceled'].agg(['mean','count'])
print(group_stats)

                       mean  count
is_repeated_guest                 
0                  0.282969  83981
1                  0.076428   3415


In [16]:
p1 = df[df['is_repeated_guest']==0]['is_canceled'].mean()
n1 = len(df[df['is_repeated_guest']==0])

p2 = df[df['is_repeated_guest']==1]['is_canceled'].mean()
n2 = len(df[df['is_repeated_guest']==1])

diff = p2 - p1

se = np.sqrt((p1*(1-p1))/n1 + (p2*(1-p2))/n2)

lower = diff - 1.96*se
upper = diff + 1.96*se

print("Difference =", diff)
print("95% CI =", (lower, upper))

Difference = -0.2065412411226042
95% CI = (np.float64(-0.21595852552257713), np.float64(-0.19712395672263125))


#R4

In [17]:
df['lead_band'] = pd.cut(
    df['lead_time'],
    bins=[0,30,90,180,737],
    labels=['0-30','31-90','91-180','180+']
)

In [18]:
pd.crosstab(df['lead_band'], df['is_repeated_guest'])

is_repeated_guest,0,1
lead_band,,
0-30,26717,1949
31-90,22560,184
91-180,18138,105
180+,11677,88


In [19]:
df.groupby(
    ['lead_band','is_repeated_guest']
)['is_canceled'].mean()

/tmp/ipykernel_700/4174408592.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


lead_band  is_repeated_guest
0-30       0                    0.194408
           1                    0.072345
31-90      0                    0.321676
           1                    0.125000
91-180     0                    0.351252
           1                    0.104762
180+       0                    0.397534
           1                    0.375000
Name: is_canceled, dtype: float64

#R5

In [20]:
# Sensitivity Analysis - Alternative 1

df['lead_band_v1'] = pd.cut(
    df['lead_time'],
    bins=[0,30,90,180,737],
    labels=['0-30','31-90','91-180','180+']
)

df.groupby(['lead_band_v1','is_repeated_guest'])['is_canceled'].mean()

/tmp/ipykernel_700/3149892239.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['lead_band_v1','is_repeated_guest'])['is_canceled'].mean()


lead_band_v1  is_repeated_guest
0-30          0                    0.194408
              1                    0.072345
31-90         0                    0.321676
              1                    0.125000
91-180        0                    0.351252
              1                    0.104762
180+          0                    0.397534
              1                    0.375000
Name: is_canceled, dtype: float64

In [21]:
# Sensitivity Analysis - Alternative 2

df['lead_band_v2'] = pd.cut(
    df['lead_time'],
    bins=[0,60,120,240,737],
    labels=['0-60','61-120','121-240','240+']
)

df.groupby(['lead_band_v2','is_repeated_guest'])['is_canceled'].mean()

/tmp/ipykernel_700/2389126448.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['lead_band_v2','is_repeated_guest'])['is_canceled'].mean()


lead_band_v2  is_repeated_guest
0-60          0                    0.235315
              1                    0.075882
61-120        0                    0.336928
              1                    0.109091
121-240       0                    0.356789
              1                    0.204082
240+          0                    0.437928
              1                    0.387755
Name: is_canceled, dtype: float64

In [22]:
# Sensitivity Analysis - Alternative 3

df.groupby(['hotel','is_repeated_guest'])['is_canceled'].mean()

hotel         is_repeated_guest
City Hotel    0                    0.306651
              1                    0.110656
Resort Hotel  0                    0.245002
              1                    0.042179
Name: is_canceled, dtype: float64

In [23]:
# Sensitivity Analysis - Alternative 4

q1 = df["lead_time"].quantile(0.25)
q3 = df["lead_time"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr

df_no_outliers = df[
    (df["lead_time"] >= lower) &
    (df["lead_time"] <= upper)
]

df_no_outliers.groupby('is_repeated_guest')['is_canceled'].mean()

,is_canceled
is_repeated_guest,
0,0.277798
1,0.074249
